# Sales ETL Pipeline

## Overview

This notebook automates the extraction, transformation, and loading (ETL) process for AdventureWorks sales data.

The pipeline:

- Reads multiple yearly sales CSV files automatically.
- Cleans and standardizes the raw sales data.
- Enriches sales records with product pricing information.
- Creates business metrics such as Revenue, Profit, and Profit Margin.
- Combines all yearly sales files into a single fact table.
- Exports the cleaned dataset for SQL loading and Power BI reporting.

---

## Workflow

```
Raw Sales Files
        │
        ▼
Read All CSV Files
        │
        ▼
Standardize Column Names
        │
        ▼
Convert Date Columns
        │
        ▼
Merge Product Information
        │
        ▼
Create Revenue & Profit Metrics
        │
        ▼
Combine All Years
        │
        ▼
Export fact_sales.csv


```
## Pipeline Architecture
```
Raw CSV Files
       │
       ▼
Python ETL
       │
       ├── Customers
       ├── Products
       ├── Territories
       ├── Calendar
       └── Sales
             │
             ▼
      MySQL Data Warehouse
             │
             ▼
      Power BI Dashboard
```

# 1. Import Required Libraries

In [86]:
import pandas as pd
from pathlib import Path
import glob
import os
import logging

# 2. Configure Logging

Configure logging to monitor the ETL process.

In [1]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

NameError: name 'logging' is not defined

In [ ]:
logging.info("Reading Sales File")

2026-07-21 05:10:22,920 - INFO - Reading Sales File


# 3. Define Project Paths

Create reusable paths

In [ ]:
project_root = Path('..')
 
raw_sales_path = project_root/ 'data' / 'raw'

cleaned_path = project_root/ 'data' / 'cleaned'


# 4. Load Product Dimension

Load the cleaned Product Dimension table.

In [ ]:
products = pd.read_csv(cleaned_path / "products_clean.csv")

logging.info("Products File Loaded Successfully")

2026-07-21 05:10:23,070 - INFO - Products File Loaded Successfully


# 5. Clean Sales Data

The `clean_sales()` function performs all transformations on each sales file.

In [ ]:
def clean_sales(sales,products):
    sales.columns = (
        sales.columns
        .str.replace(" ","_")
        .str.replace(r'(?<!^)(?=[A-Z])',"_",regex=True)
        .str.lower()
    )


    date_columns = ['order_date',"stock_date"]

    for col in date_columns:
        sales[col] = pd.to_datetime(sales[col],errors='coerce')

    sales = sales.merge(
    products[['product_key',"product_price",'product_cost']],
    on='product_key',
    how='left'
)
    
    sales['revenue'] = sales['product_price'] * sales['order_quantity']

    sales['profit'] = (
    (sales['product_price'] - sales['product_cost'])
    * sales['order_quantity']
)
    sales['profit_margin'] = (
    sales['profit'] / sales['revenue']
)*100


    return sales

# 6.  Process Multiple Sales Files

Automatically locate all yearly sales files.

In [ ]:
sales_files = glob.glob(str(raw_sales_path/'AdventureWorks_Sales_*.csv'))



# 7. Error Handling

Each file is processed inside a `try-except` block.

In [ ]:
sales_list = []

for file in sales_files:

    try:

        logging.info(f"Reading {os.path.basename(file)}")

        df = pd.read_csv(file)

        df = clean_sales(df,products)

        sales_list.append(df)

        logging.info("File Loaded Successfully")

    except Exception as e:

        logging.error(f"Error in {os.path.basename(file)}")

        logging.error(e)

2026-07-21 05:10:23,179 - INFO - Reading AdventureWorks_Sales_2015.csv
2026-07-21 05:10:23,216 - INFO - File Loaded Successfully
2026-07-21 05:10:23,219 - INFO - Reading AdventureWorks_Sales_2016.csv
2026-07-21 05:10:23,430 - INFO - File Loaded Successfully
2026-07-21 05:10:23,465 - INFO - Reading AdventureWorks_Sales_2017.csv
2026-07-21 05:10:23,707 - INFO - File Loaded Successfully


# 8. Combine Sales Data

Merge all cleaned yearly datasets into one consolidated fact table.

In [ ]:
fact_sales = pd.concat(sales_list,ignore_index=True)
logging.info("All Sales Files Combined Successfully")

2026-07-21 05:10:23,730 - INFO - All Sales Files Combined Successfully


# 9. Export Clean Dataset

In [ ]:
fact_sales.to_csv(
    cleaned_path / "fact_sales.csv",
    index=False
)